<a href="https://colab.research.google.com/github/diegovc1987-boop/Ejercicio4/blob/main/Ejercicio4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!pip install gradio transformers google-generativeai

In [4]:
import gradio as gr
from transformers import pipeline
import google.generativeai as genai
from google.colab import userdata

# --- 1. Sentiment Analyzer Tab ---
sentiment_analyzer = pipeline("sentiment-analysis")

def analyze_sentiment(text):
    result = sentiment_analyzer(text)[0]
    return f"Sentiment: {result['label']}, Score: {result['score']:.2f}"

sentiment_interface = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(lines=5, label="Enter text for sentiment analysis"),
    outputs="text",
    title="Sentiment Analyzer",
    description="Analyze the sentiment (positive/negative) of your text."
)

# --- 2. Text Summarizer Tab (using Gemini) ---
# Ensure you have your Google API Key set in Colab secrets as 'GOOGLE_API_KEY'
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-1.5-flash') # Using a fast model for summarization
    print("Gemini API configured successfully.")
except Exception as e:
    print(f"Error configuring Gemini API: {e}. Please ensure 'GOOGLE_API_KEY' is set in Colab secrets.")
    gemini_model = None

def summarize_text(text):
    if not gemini_model:
        return "Gemini API is not configured. Please check your API key."
    if not text.strip():
        return "Please provide text to summarize."
    try:
        response = gemini_model.generate_content(f"""Summarize the following text:

{text}""")
        return response.text
    except Exception as e:
        return f"Error during summarization: {e}"

summarizer_interface = gr.Interface(
    fn=summarize_text,
    inputs=gr.Textbox(lines=10, label="Enter text to summarize"),
    outputs="text",
    title="Text Summarizer (powered by Gemini)",
    description="Summarize long texts using the Gemini AI model."
)

# --- 3. Translation Tab ---
# Using a small English to Spanish model for demonstration
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-es")

def translate_text(text):
    if not text.strip():
        return "Please provide text to translate."
    try:
        result = translator(text)[0]['translation_text']
        return result
    except Exception as e:
        return f"Error during translation: {e}"

translation_interface = gr.Interface(
    fn=translate_text,
    inputs=gr.Textbox(lines=5, label="Enter text to translate (English to Spanish)"),
    outputs="text",
    title="English to Spanish Translator",
    description="Translate text from English to Spanish."
)

# --- Combine all interfaces into a TabbedInterface ---
app = gr.TabbedInterface([
    sentiment_interface,
    summarizer_interface,
    translation_interface
],
names=["Sentiment Analysis", "Text Summarization", "Translation"])

app.launch(debug=True)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Gemini API configured successfully.


KeyError: "Invalid translation task translation, use 'translation_XX_to_YY' format"